# Day 24 — Practice Session · **SOLUTIONS**
### The Normal Distribution · Statistics & Probability

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day24_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(124)

# Heights: genuinely normal
heights = pd.Series(rng.normal(170, 10, 2000))

# Delivery times: skewed, with a floor at zero
delivery = pd.Series(rng.gamma(2, 12, 2000))

print(f"heights  n={len(heights)}  mean {heights.mean():.1f}  sd {heights.std():.1f}")
print(f"delivery n={len(delivery)}  mean {delivery.mean():.1f}  sd {delivery.std():.1f}")

---
## Setup — run this first

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(124)

# Heights: genuinely normal
heights = pd.Series(rng.normal(170, 10, 2000))

# Delivery times: skewed, with a floor at zero
delivery = pd.Series(rng.gamma(2, 12, 2000))

print(f"heights  n={len(heights)}  mean {heights.mean():.1f}  sd {heights.std():.1f}")
print(f"delivery n={len(delivery)}  mean {delivery.mean():.1f}  sd {delivery.std():.1f}")

---
# Part A — Predict the Output

**Teaching note:** A3 is the one that catches everyone (forgotten `loc`/`scale`). A7 is the
comparison that makes z-scores click.

### A1. What are the three bands?

In [ ]:
for k in (1, 2, 3):
    print(round((stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100, 2))

*Your prediction:*  

> ### ✅ Answer A1
> ```
> 68.27
> 95.45
> 99.73
> ```
> The empirical rule, exactly. `cdf(k)` gives everything below +k; subtracting `cdf(-k)`
> removes everything below −k, leaving the band between them.
>
> Memorise the rounded form — **68, 95, 99.7** — and the "outside" version:
> beyond 2σ is about **1 in 22**, beyond 3σ is about **1 in 370**.

### A2. How many standard deviations out is this?

In [ ]:
# commute: mean 32 min, sd 6
print((50 - 32) / 6)
print(round((1 - stats.norm.cdf((50 - 32) / 6)) * 100, 3))

*Your prediction:*  

> ### ✅ Answer A2
> ```
> 3.0
> 0.135
> ```
> A 50-minute journey is **3σ** above the mean, and about **0.135%** of journeys are that
> slow or worse — roughly **1 in 740**.
>
> Note that is the *one-sided* figure. The familiar "1 in 370" counts both tails; here you
> only care about being late, so you halve it.

### A3. What does this give?

In [ ]:
# IQ has mean 100 and sd 15
print(round(stats.norm.cdf(130), 6))
print(round(stats.norm.cdf(130, 100, 15), 4))

*Your prediction:*  

> ### ✅ Answer A3
> ```
> 1.0
> 0.9772
> ```
> **The first line is the bug everyone writes once.** With no `loc` and `scale`, scipy assumes
> the *standard* normal — mean 0, sd 1 — so it is asking "what fraction is below 130 standard
> deviations?" The answer rounds to 1.0.
>
> The second passes the real parameters and gives the correct **97.72%**.
>
> **Always pass `loc` and `scale`**, or standardise first and use the bare version.

### A4. Are these the same?

In [ ]:
print(round(stats.norm.cdf(130, 100, 15), 4))
print(round(stats.norm.cdf((130 - 100) / 15), 4))

*Your prediction:*  

> ### ✅ Answer A4
> ```
> 0.9772
> 0.9772
> ```
> **Identical** — and this is the whole point of standardising. Converting to a z-score first
> and then using the standard normal gives exactly the same answer as passing the parameters.
>
> It is why one table of probabilities worked for every normal distribution in the days before
> computers, and why z-scores are worth learning at all.

### A5. What values come back?

In [ ]:
print(round(stats.norm.ppf(0.975), 4))
print(round(stats.norm.ppf(0.95), 4))
print(round(stats.norm.ppf(0.5), 4))

*Your prediction:*  

> ### ✅ Answer A5
> ```
> 1.96
> 1.6449
> 0.0
> ```
> **1.96** leaves 2.5% in each tail, so 95% sits in the middle — that is where "95% confidence
> interval" comes from, and you will meet it in every formula from Day 29.
>
> **1.645** leaves 5% in one tail, used for one-sided tests.
>
> `ppf(0.5)` is 0 because the standard normal is symmetric about zero: half the curve is below
> the mean.

### A6. Does the rule hold here?

In [ ]:
d = pd.Series(rng.gamma(2, 12, 50000))   # skewed, floor at zero
m, sd = d.mean(), d.std()
print(round(d.between(m - sd, m + sd).mean() * 100, 1))
print(round((d < m).mean() * 100, 1))
print(round(m - 2 * sd, 1))

*Your prediction:*  

> ### ✅ Answer A6
> Approximately:
> ```
> 73.9
> 57.7
> -24.0
> ```
> *(Run it — the exact figures depend on the draw.)*
>
> **Three failures at once.** The 1σ band holds about 74% rather than 68%. Only 58% sit below
> the mean, not 50%. And `mean − 2sd` is **negative**, which is impossible for a delivery time.
>
> That last one is the giveaway: a normal distribution has tails running to infinity in *both*
> directions, so fitting one to data with a hard floor at zero always puts real probability
> below zero.

### A7. Which performance is better?

In [ ]:
# Maths: scored 78, class mean 70, sd 8
# Physics: scored 65, class mean 55, sd 5
print((78 - 70) / 8)
print((65 - 55) / 5)

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 1.0
> 2.0
> ```
> **Physics is the better performance, despite the lower raw mark.**
>
> 78 in Maths beats about 84% of that class. 65 in Physics beats about **98%** of that one.
>
> This is the single most useful thing z-scores do: they let you compare performances measured
> on scales that are not otherwise comparable. Without standardising, 78 simply looks bigger
> than 65.

### A8. What does the p-value mean?

In [ ]:
small = rng.normal(0, 1, 20)
stat, p = stats.shapiro(small)
print(round(p, 3) > 0.05)

*Your prediction:*  

> ### ✅ Answer A8
> ```
> True
> ```
> **But be careful what you conclude.** `p > 0.05` means *no evidence against normality* — it
> does **not** mean the data is normal.
>
> With only 20 points, a Shapiro test has very little power: it would fail to detect quite
> serious non-normality. A "pass" on a small sample is almost uninformative.
>
> And the reverse problem appears on large data: on 50,000 rows the test rejects normality for
> a trivial, meaningless deviation. **On large data, trust the plot.**

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 is the most common scipy error in the whole module. B4 is the
conceptual one.

### B1. What fraction of students score above 85?

In [ ]:
# marks: mean 72, sd 11
print(f"{(1 - stats.norm.cdf(85)) * 100:.2f}% score above 85")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** prints `0.00%`, which is obviously wrong.
> **Cause:** `loc` and `scale` were not passed, so scipy used the standard normal and asked
> "what fraction is above 85 *standard deviations*?"

In [ ]:
mu, sd = 72, 11

print("the bug:")
print(f"  {(1 - stats.norm.cdf(85)) * 100:.4f}%   <- assumes mean 0, sd 1")
print()
print("two correct ways:")
print(f"  pass the parameters : {(1 - stats.norm.cdf(85, mu, sd)) * 100:.2f}%")
print(f"  standardise first   : {(1 - stats.norm.cdf((85 - mu) / sd)) * 100:.2f}%")

# Both give 11.86%. Pick one style and stay with it - mixing them is how
# the bug creeps in.

### B2. Find the mark for the top 10%.

In [ ]:
mu, sd = 72, 11
print(round(stats.norm.ppf(0.10, mu, sd), 1))

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B2
> **Symptom:** returns about **57.9**, which is a *low* mark — clearly not the top 10%.
> **Cause:** `ppf(0.10)` finds the value with 10% **below** it. The top 10% starts where 90%
> is below.

In [ ]:
mu, sd = 72, 11

print(f"ppf(0.10) = {stats.norm.ppf(0.10, mu, sd):.1f}   <- the BOTTOM 10% ends here")
print(f"ppf(0.90) = {stats.norm.ppf(0.90, mu, sd):.1f}   <- the TOP 10% starts here")
print()
print("sanity check:")
print(f"  fraction above 86.1: {(1 - stats.norm.cdf(86.1, mu, sd)) * 100:.1f}%")

# ppf ALWAYS works from the bottom up. For a top-k% question, ask for
# ppf(1 - k/100).

### B3. Compare two students across different exams.

In [ ]:
# Ravi: 78 in Maths   (class mean 70, sd 8)
# Sara: 80 in History (class mean 76, sd 16)
print("Sara did better:", 80 > 78)

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** concludes Sara did better because 80 > 78.
> **Cause:** the two marks come from different distributions. Comparing raw marks across exams
> with different means and spreads is meaningless.

In [ ]:
people = pd.DataFrame({
    "student": ["Ravi", "Sara"],
    "exam":    ["Maths", "History"],
    "mark":    [78, 80],
    "mean":    [70, 76],
    "sd":      [8, 16],
})
people["z"] = (people["mark"] - people["mean"]) / people["sd"]
people["beat_pct"] = (stats.norm.cdf(people["z"]) * 100).round(1)
print(people.to_string(index=False))
print()
print("Ravi's z is 1.00; Sara's is 0.25. RAVI did better.")
print("The raw marks pointed the other way because History was marked")
print("more generously AND had twice the spread.")

### B4. Check whether delivery times are normal.

In [ ]:
m, sd = delivery.mean(), delivery.std()
within2 = delivery.between(m - 2*sd, m + 2*sd).mean() * 100
print(f"{within2:.1f}% within 2 sd - close to 95%, so it is normal")

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** a single check passes, and the conclusion is drawn from it alone.
> **Cause:** the 2σ band is the **least sensitive** of the three to skew. It can pass on data
> that is badly non-normal — exactly what happened with the Day 23 salaries.

In [ ]:
m, sd = delivery.mean(), delivery.std()

print(f"{'check':<20} {'normal says':>12} {'reality':>10}")
for k in (1, 2, 3):
    obs = delivery.between(m - k*sd, m + k*sd).mean() * 100
    exp = (stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100
    print(f"{'within ' + str(k) + ' sd':<20} {exp:>11.1f}% {obs:>9.1f}%")
print(f"{'below the mean':<20} {50.0:>11.1f}% {(delivery < m).mean()*100:>9.1f}%")
print(f"{'skew':<20} {0.0:>11.1f}  {delivery.skew():>9.2f}")
print()
print(f"and mean - 2sd = {m - 2*sd:.1f}  <- a negative delivery time")
print()
print("The 2 sd check is nearly right. EVERYTHING ELSE fails.")
print("One passing check is not evidence - test several, and plot it.")

---
# Part C — Write the Code

**Teaching note:** C5 is the reusable function; C3 is the one they will actually need in the
Module 2 project.

### C1. Check the bands
For `heights`, print the observed percentage within 1, 2 and 3 standard deviations next to
what the normal distribution predicts.

In [ ]:
# your code here

In [ ]:
m, sd = heights.mean(), heights.std()

print(f"{'band':<10} {'observed':>10} {'predicted':>11}")
for k in (1, 2, 3):
    obs = heights.between(m - k*sd, m + k*sd).mean() * 100
    exp = (stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100
    print(f"{str(k) + ' sd':<10} {obs:>9.1f}% {exp:>10.2f}%")

print()
print(f"below the mean: {(heights < m).mean()*100:.1f}%   (normal says 50.0%)")
print(f"skew: {heights.skew():+.3f}")
print()
print("All three bands land within about a percentage point, and the data")
print("splits evenly about the mean. This one really is normal.")

### C2. Standardise and compare
Three students sat different exams. Work out who performed best.

| Student | Exam | Mark | Class mean | Class SD |
|---|---|---|---|---|
| Ravi | Maths | 78 | 70 | 8 |
| Sara | Physics | 65 | 55 | 5 |
| Amit | History | 88 | 80 | 12 |

In [ ]:
# your code here

In [ ]:
exams = pd.DataFrame({
    "student": ["Ravi", "Sara", "Amit"],
    "exam":    ["Maths", "Physics", "History"],
    "mark":    [78, 65, 88],
    "mean":    [70, 55, 80],
    "sd":      [8, 5, 12],
})
exams["z"] = ((exams["mark"] - exams["mean"]) / exams["sd"]).round(3)
exams["beat_pct"] = (stats.norm.cdf(exams["z"]) * 100).round(1)

print(exams.sort_values("z", ascending=False).to_string(index=False))
print()
print("SARA performed best, with the LOWEST raw mark of the three.")
print("Her 65 beat 97.7% of her class; Amit's 88 beat only 74.8% of his.")

# Ranking by raw mark gives Amit, Ravi, Sara - exactly backwards.

### C3. Percentiles both ways
For `heights` (mean ≈ 170, sd ≈ 10): what height is the 90th percentile, and what percentage
of people are taller than 185 cm? Compute each from the normal model AND from the data, then
compare.

In [ ]:
# your code here

In [ ]:
m, sd = heights.mean(), heights.std()

# --- the 90th percentile
model_p90 = stats.norm.ppf(0.90, m, sd)
real_p90  = heights.quantile(0.90)
print(f"90th percentile   model {model_p90:7.2f}   data {real_p90:7.2f}   "
      f"diff {abs(model_p90 - real_p90):.2f}")

# --- the fraction above 185
model_above = (1 - stats.norm.cdf(185, m, sd)) * 100
real_above  = (heights > 185).mean() * 100
print(f"above 185 cm      model {model_above:6.2f}%   data {real_above:6.2f}%   "
      f"diff {abs(model_above - real_above):.2f}pp")

print()
print("They agree closely BECAUSE this data really is normal.")
print("Run the same comparison on `delivery` and the two answers diverge badly -")
print("that gap is exactly the cost of a wrong distributional assumption.")

### C4. Show the rule failing
Produce a table for `delivery` showing, for each of the three bands, what the normal
distribution predicts against what actually happens. Add one line saying why it fails.

In [ ]:
# your code here

In [ ]:
m, sd = delivery.mean(), delivery.std()

rows = []
for k in (1, 2, 3):
    rows.append({
        "band":      f"{k} sd",
        "normal":    round((stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100, 1),
        "observed":  round(delivery.between(m - k*sd, m + k*sd).mean() * 100, 1),
        "lower_end": round(m - k*sd, 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

print()
print(f"skew {delivery.skew():.2f}   below the mean {(delivery < m).mean()*100:.1f}%")
print()
print("WHY IT FAILS: delivery times have a hard floor at zero and a long right")
print("tail. A normal curve is symmetric with infinite tails both ways, so")
print("fitting one here puts real probability on NEGATIVE delivery times -")
print("see the lower_end column turn negative at 2 sd.")

### C5. A normality check you can trust
Write `normal_check(s)` that tests **several** things — the bands, the split about the mean,
the skew — and returns a verdict. Run it on both `heights` and `delivery`.

In [ ]:
# your code here

In [ ]:
def normal_check(s, name="value"):
    """Several independent checks - never just one."""
    s = s.dropna()
    m, sd = s.mean(), s.std()

    bands = {}
    for k in (1, 2, 3):
        obs = s.between(m - k*sd, m + k*sd).mean() * 100
        exp = (stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100
        bands[f"within_{k}sd"] = f"{obs:.1f} vs {exp:.1f}"

    one_sd = s.between(m - sd, m + sd).mean() * 100
    below  = (s < m).mean() * 100

    out = {
        "n":          len(s),
        "skew":       round(s.skew(), 2),
        "below_mean": round(below, 1),
        "lower_2sd":  round(m - 2*sd, 1),
        **bands,
    }
    bad = (abs(out["skew"]) > 1
           or abs(below - 50) > 8
           or abs(one_sd - 68.3) > 6)
    out["verdict"] = "NOT normal" if bad else "close enough"
    return pd.Series(out, name=name)


print(pd.concat([normal_check(heights, "heights"),
                 normal_check(delivery, "delivery")], axis=1))

# THREE independent signals, any one of which can fire. That is what stops
# a coincidental pass on one band from fooling you.

---
# Part D — Challenge

**Teaching note:** D1 is the one that makes the coincidence lesson stick, because they build
it themselves.

### D1. Break the rule yourself
Build a skewed dataset. Show that **at least one** band test passes while the distribution is
clearly not normal — and explain what gives it away.

In [ ]:
# your code here

In [ ]:
sim = np.random.default_rng(2024)
inc = np.r_[sim.normal(45000, 8000, 950), sim.normal(400000, 60000, 50)]
sal = pd.Series(inc)
m, sd = sal.mean(), sal.std()

print(f"skew {sal.skew():.2f}   - obviously not normal\n")
print(f"{'check':<18} {'normal says':>12} {'reality':>10}   verdict")
for k in (1, 2, 3):
    obs = sal.between(m - k*sd, m + k*sd).mean() * 100
    exp = (stats.norm.cdf(k) - stats.norm.cdf(-k)) * 100
    ok = "PASSES" if abs(obs - exp) < 1 else "fails"
    print(f"{'within ' + str(k) + ' sd':<18} {exp:>11.1f}% {obs:>9.1f}%   {ok}")
print(f"{'below the mean':<18} {50.0:>11.1f}% {(sal < m).mean()*100:>9.1f}%   fails")

print()
print("THE 2 SD CHECK PASSES - 95.0% against a predicted 95.4%.")
print("If that were the only thing you tested you would have concluded the")
print("data was normal and moved on. It is a coincidence.")
print()
print("WHAT GIVES IT AWAY:")
print(f"  1 sd band out by {abs(sal.between(m-sd,m+sd).mean()*100 - 68.27):.0f} percentage points")
print(f"  {(sal < m).mean()*100:.0f}% sit below the mean, not 50%")
print(f"  mean - 2sd = {m - 2*sd:,.0f}, a negative salary")
print(f"  skew {sal.skew():.2f}, far above 1")
print()
print("ONE PASSING CHECK IS NOT EVIDENCE.")

### D2. Set an honest threshold
A company wants an alert when a delivery is "unusually slow" — the slowest 5%. Compute that
threshold from the normal model and from the data. Which should they use, and what does the
wrong choice cost?

In [ ]:
# your code here

In [ ]:
m, sd = delivery.mean(), delivery.std()

model_cut = m + 1.645 * sd           # the normal 95th percentile
real_cut  = delivery.quantile(0.95)  # the actual one

print(f"normal model threshold : {model_cut:6.1f} minutes")
print(f"actual 95th percentile : {real_cut:6.1f} minutes")
print(f"difference             : {abs(model_cut - real_cut):6.1f} minutes")
print()

for name, cut in [("model", model_cut), ("actual", real_cut)]:
    flagged = (delivery > cut).mean() * 100
    print(f"using the {name:<7} threshold flags {flagged:5.2f}% of deliveries")

print()
print("THE COST: they asked for the slowest 5%. The normal model's threshold")
print(f"flags about {(delivery > model_cut).mean()*100:.1f}% instead.")
print()
print("USE THE EMPIRICAL PERCENTILE. quantile(0.95) makes no assumption about")
print("the shape - it just reads the number off the data you have. The normal")
print("model is only worth using when you have CHECKED that it fits.")

---
## Done? Self-check

- [ ] I know the three bands and the "1 in 22 / 1 in 370" form
- [ ] I always pass `loc` and `scale` to scipy
- [ ] I know `cdf` and `ppf` are inverses, and which direction each goes
- [ ] I can standardise two marks and say which is better
- [ ] I know where 1.96 comes from
- [ ] I check more than one thing before assuming normality
- [ ] I know `p > 0.05` does not mean "it is normal"

### Homework
1. Run `normal_check` on every numeric column of your Day 21 project data.
2. Find your own two-exam example and compute both z-scores.
3. Use `ppf` to find the mark that would put a student in the top 5%.

### Next class — Topic 2.5: Binomial & Poisson
The distributions for **counting** things.

---
*Prepared by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"`stats.norm.cdf(130)` — what did I forget?"* → `loc` and `scale`.
2. *"78 in Maths or 65 in Physics — which is better?"* → depends on the z-scores.
3. *"The 2σ check passed. Is the data normal?"* → not necessarily. Check more.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "Most real data is normal" | Much of it is not — income, delays, file sizes |
| "`cdf(130)` works without parameters" | It assumes mean 0, sd 1 |
| "`ppf(0.10)` gives the top 10%" | It gives the **bottom** 10% |
| "A higher raw mark is a better performance" | Not across different distributions |
| "One band check is enough" | The 2σ band is the least sensitive to skew |
| "`p > 0.05` means normal" | It means **no evidence against** |
| "z-scores are useless on skewed data" | They still **rank** — they just stop giving probabilities |

**D1 is the centrepiece.** Students build a dataset with skew 4.3 and watch the 2σ check pass
at 95.0% against a predicted 95.4%. Having constructed the counterexample themselves, they
stop treating a single passing check as proof. Several will want to know whether that is a
fluke of this dataset — it is not; the 2σ band is genuinely the least sensitive of the three
to skew, which is worth saying.

**B1 will happen in the room before you mention it.** Someone will write `stats.norm.cdf(85)`
and get a number that is obviously wrong. That is the best possible moment to explain `loc`
and `scale`.

**D2 is the practical payoff.** The normal model and the empirical percentile give different
thresholds, and the gap is a real operational error. The lesson is simple: `quantile(0.95)`
makes no assumption about shape, so prefer it unless you have checked that the normal fits.

**Homework given:** `normal_check` on their Day 21 columns; their own two-exam comparison;
`ppf` for a top-5% cut-off.

**Next session:** Topic 2.5 — binomial and Poisson, moving from measuring continuous
quantities to **counting** events.